In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-4B-Instruct-2507"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name_14b = "Qwen/Qwen2.5-14B-Instruct"

model_14b = AutoModelForCausalLM.from_pretrained(
    model_name_14b,
    dtype="auto",
    device_map="auto"
)
tokenizer_14b = AutoTokenizer.from_pretrained(model_name_14b)

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [2]:
import xml.etree.ElementTree as ET

dev_tree = ET.parse('./data/dev/archehr-qa.xml')
test_tree = ET.parse('./data/test/archehr-qa.xml')

dev_root = dev_tree.getroot()
test_root = test_tree.getroot()

In [3]:
dev_patient = [i.text.strip() for i in dev_root.iter('patient_narrative')]
dev_clinician = [i.text.strip() for i in dev_root.iter('clinician_question')]

test_patient = [i.text.strip() for i in test_root.iter('patient_narrative')]
test_clinician = [i.text.strip() for i in test_root.iter('clinician_question')]

print(len(dev_patient), len(dev_clinician), len(test_patient), len(test_clinician))

20 20 100 100


## Evaluations

In [4]:
import numpy as np
import torch
from evaluate import load
from enum import Enum

class BertScorer:
    def __init__(self, device):
        self.device = device
        self.bertscore = load("bertscore")

    def compute_scores(self, references, predictions):
        scores = self.bertscore.compute(
            references=references,
            predictions=predictions,
            model_type="distilbert-base-uncased",
            device=self.device,
            num_layers=4,
            batch_size=8,
            nthreads=4,
            all_layers=False,
            idf=False,
            lang="en",
            rescale_with_baseline=True,
            baseline_path=None,
        )
        return scores["f1"]

    def compute_overall_score(self, references, predictions):
        scores = self.compute_scores(references, predictions)
        return np.mean(scores)


class RougeType(Enum):
    ROUGE1 = "rouge1"
    ROUGE2 = "rouge2"
    ROUGEL = "rougeL"
    ROUGELSUM = "rougeLsum"


class RougeScorer:
    def __init__(self, rouges=["rouge1", "rouge2", "rougeL", "rougeLsum"]):
        self.rouge = load("rouge")
        self.rouge_types = [RougeType(rt) for rt in rouges]

    def compute_scores(self, references, predictions):
        scores = {rt.value: [] for rt in self.rouge_types}
        for r, p in zip(references, predictions):
            rouge_scores = self.rouge.compute(references=[r], predictions=[p])
            for rouge_type, rt_scores in scores.items():
                rt_scores.append(rouge_scores[rouge_type])
        return scores

    def compute_overall_score(self, references, predictions):
        scores = self.compute_scores(references, predictions)
        return {key: np.mean(value) for key, value in scores.items()}

device = "mps"

rouge_scorer = RougeScorer()
bert_scorer = BertScorer(device)

In [ ]:
# bert_score = bert_scorer.compute_overall_score(truth, initial_answers)
# rouge_score = rouge_scorer.compute_overall_score(truth, initial_answers)

## One prompt

### 4B

#### Prompt 1

In [18]:
# prepare the model input
SYSTEM_PROMPT = f"""
Your task to transform a free-text, patient-authored question into a clear and concise clinician-interpreted question that reflects how a clinician would query a smart electronic health record (EHR) system when preparing a response to the patient.
Your outputs should be very brief only focusing on the key question in less than 15 words.

Goal:
    Generate a single, well-formed clinician question that captures the core clinical information need implied by the patient’s narrative, phrased as a query to an intelligent EHR system.
Input:
    Patient‑authored question (Patient Question)
Expected output:
    Clinician‑interpreted question that captures the main medical concern(s) (≤ 15 words).

Examples:
    Patient Question: {dev_patient[0]}
    Clinician-interpreted Question: {dev_clinician[0]}

    Patient Question: {dev_patient[1]}
    Clinician-interpreted Question: {dev_clinician[1]}

    Patient Question: {dev_patient[2]}
    Clinician-interpreted Question: {dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:{question}\nClinician-interpreted Question:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=30,
        temperature=0.01
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [19]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

#### Prompt 2

In [6]:
# prepare the model input
SYSTEM_PROMPT = f"""
You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:{question}\nClinician-interpreted Question:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [7]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [8]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


0.3843627147376537 {'rouge1': np.float64(0.28541264875065714), 'rouge2': np.float64(0.12945063993977038), 'rougeL': np.float64(0.25633039766840604), 'rougeLsum': np.float64(0.25633039766840604)}


### 14B

#### Prompt 1

In [7]:
# prepare the model input
SYSTEM_PROMPT = f"""
Your task to transform a free-text, patient-authored question into a clear and concise clinician-interpreted question that reflects how a clinician would query a smart electronic health record (EHR) system when preparing a response to the patient.
Your outputs should be very brief only focusing on the key question in less than 15 words.

Goal:
    Generate a single, well-formed clinician question that captures the core clinical information need implied by the patient’s narrative, phrased as a query to an intelligent EHR system.
Input:
    Patient‑authored question (Patient Question)
Expected output:
    Clinician‑interpreted question that captures the main medical concern(s) (≤ 15 words).

Examples:
    Patient Question: {dev_patient[0]}
    Clinician-interpreted Question: {dev_clinician[0]}

    Patient Question: {dev_patient[1]}
    Clinician-interpreted Question: {dev_clinician[1]}

    Patient Question: {dev_patient[2]}
    Clinician-interpreted Question: {dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:{question}\nClinician-interpreted Question:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [8]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [10]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


0.39582759514451027 {'rouge1': np.float64(0.2785342995598263), 'rouge2': np.float64(0.10469718433105162), 'rougeL': np.float64(0.26230719333272), 'rougeLsum': np.float64(0.26230719333272)}


#### Prompt 2

In [11]:
# prepare the model input
SYSTEM_PROMPT = f"""
You are rewriting patient-authored questions into a single, concise clinician-interpreted question.

Task:
- Identify the main information need.
- Rewrite it as one short question a clinician would ask internally.
- Focus on intent, not medical detail.

Rules:
- Output ONE sentence only
- No explanations or background
- Do NOT add medical terminology
- Keep it as short and simple as possible (≤15 words)
- Prefer "Why", "What", "When", or "Did"

Examples:

Patient Question:
{dev_patient[0]}
Clinician-interpreted Question:
{dev_clinician[0]}

Patient Question:
{dev_patient[1]}
Clinician-interpreted Question:
{dev_clinician[1]}

Patient Question:
{dev_patient[2]}
Clinician-interpreted Question:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient Question:\n{question}\n\nClinician-interpreted Question:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [13]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

In [14]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

0.39450974985957143 {'rouge1': np.float64(0.2747667494291645), 'rouge2': np.float64(0.09732247284878864), 'rougeL': np.float64(0.25672327116829496), 'rougeLsum': np.float64(0.25672327116829496)}


#### Prompt 3

In [19]:
# prepare the model input
SYSTEM_PROMPT = f"""
You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [20]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

In [21]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

0.4100103175267577 {'rouge1': np.float64(0.31137120510125604), 'rouge2': np.float64(0.12588216867934865), 'rougeL': np.float64(0.2948975354017799), 'rougeLsum': np.float64(0.2948975354017799)}


## Multi Prompt

### 14B initial (prompt 3) + 4B hallucination detection

In [6]:
# prepare the model input
SYSTEM_PROMPT = f"""You are converting long, messy questions into their single core question.

Task:
- Identify what the person is really asking.
- Rewrite it as ONE short, clear question.
- Remove extra details, emotion, and background.

Rules:
- One sentence only
- Keep it as short as possible
- Do NOT explain or summarize
- Do NOT add new details
- Prefer simple "Why / What / Did" questions
- Output ONLY the final question. No other text.

Examples:

Input:
{dev_patient[0]}
Output:
{dev_clinician[0]}

Input:
{dev_patient[1]}
Output:
{dev_clinician[1]}

Input:
{dev_patient[2]}
Output:
{dev_clinician[2]}
"""

def get_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Input:\n{question}\n\nOutput:"},
    ]
    text = tokenizer_14b.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_14b([text], return_tensors="pt").to(model_14b.device)
    
    # conduct text completion
    generated_ids = model_14b.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer_14b.decode(output_ids, skip_special_tokens=True)

    return content

In [7]:
# prepare the model input
CLEANING_PROMPT = f"""You are tasked with detecting and removing hallucinations from an LLM output.
You will be given outputs from an LLM which should just contain a concise question.
If the output fits this criteria, simply return the same output.
If the output contains explanations, unnecessary content, hallucinations, etc. remove them and only return the actual question.
Do NOT explain or summarize
Do NOT add new details
"""

def get_clear_answer(question):
    messages = [
        {"role": "system", "content": CLEANING_PROMPT},
        {"role": "user", "content": f"LLM Output:\n{question}\n\nCleared Output:"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=30,
        do_sample=False
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    content = tokenizer.decode(output_ids, skip_special_tokens=True)

    return content

In [8]:
dev_answers = []
for q in dev_patient:
    a = get_answer(q)
    dev_answers.append(a)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [9]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, dev_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, dev_answers)

print(bert_score, rouge_score)

/Users/sebis/.pyenv/versions/3.12.2/envs/workshop/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self._baseline_vals = torch.from_numpy(


0.3837074749171734 {'rouge1': np.float64(0.30993198531720867), 'rouge2': np.float64(0.12588216867934865), 'rougeL': np.float64(0.28667670642233023), 'rougeLsum': np.float64(0.28667670642233023)}


In [12]:
clear_answers = []
for q in dev_answers:
    a = get_clear_answer(q)
    clear_answers.append(a)

In [13]:
bert_score = bert_scorer.compute_overall_score(dev_clinician, clear_answers)
rouge_score = rouge_scorer.compute_overall_score(dev_clinician, clear_answers)

print(bert_score, rouge_score)

0.41940785348415377 {'rouge1': np.float64(0.31064968866649095), 'rouge2': np.float64(0.12588216867934865), 'rougeL': np.float64(0.2873944097716125), 'rougeLsum': np.float64(0.2873944097716125)}


In [15]:
test_answers = []
for q in test_patient:
    a = get_answer(q)
    test_answers.append(a)

clear_test_answers = []
for q in test_answers:
    a = get_clear_answer(q)
    clear_test_answers.append(a)

In [16]:
bert_score = bert_scorer.compute_overall_score(test_clinician, test_answers)
rouge_score = rouge_scorer.compute_overall_score(test_clinician, test_answers)

print(bert_score, rouge_score)

bert_score = bert_scorer.compute_overall_score(test_clinician, clear_test_answers)
rouge_score = rouge_scorer.compute_overall_score(test_clinician, clear_test_answers)

print(bert_score, rouge_score)

0.34550071209669114 {'rouge1': np.float64(0.22855740062867724), 'rouge2': np.float64(0.04575583437723148), 'rougeL': np.float64(0.19742719788581164), 'rougeLsum': np.float64(0.19801543317992934)}
0.3476670618355274 {'rouge1': np.float64(0.22917885217975506), 'rouge2': np.float64(0.04575583437723148), 'rougeL': np.float64(0.1991774252715477), 'rougeLsum': np.float64(0.1991774252715477)}
